In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_PATH = Path.cwd().parent
RELATIV_PATH_PROCESSED_DATA = "data/processed/processed_dataset_2023.csv"
RELATIV_PATH_INTERIM_DATA = "data/interim/interim_dataset.csv"

# Standardization and extreme values before clustering

## Decisions from notebooks 04 and 05: features selected for clustering (main version, 162 countries, 2023)

| Feature | Role | Log | Standardization |
|---|---|---|---|
| GDP per capita (PPP) | Active | yes | yes, after log |
| Life expectancy | Active | no | yes |
| CO₂ per capita | Active | yes | yes, after log |
| PM2.5 | Active | yes | yes, after log |
| Urbanization (% of population) | Active | no | yes |
| Population | Illustrative | no | no |
| Land area | Illustrative | no | no |
| World Bank region, income group (to be confirmed) | Illustrative (later) | no | no |
| Health spending (% of GDP) | Removed from main version (variant, clustering stage) | no | n/a |
| Health spending per capita | Removed (built from GDP per capita) | n/a | n/a |

_Order: log first, then standardization. Standardization is applied at modeling time, not stored in `processed/`._

In [2]:
MIN_POPULATION = 300000
COLUMNS_WITH_LOG = [
    "gdp_per_capita_ppp_current_ppp",
    "co2_emissions_per_capita_t",
    "pm2_5_exposure_mcg_m3",
]
Z_THRESHOLD = 2.5


def prepare_data(min_population=MIN_POPULATION, excluded_areas=()):
    """Load the processed dataset, exclude micro-states and listed countries, add log columns (raw columns are kept for the variants)."""
    df = pd.read_csv(PROJECT_PATH / RELATIV_PATH_PROCESSED_DATA)
    df = df[(df["population_ps"] >= min_population) & ~df["ref_area"].isin(excluded_areas)].copy()

    # The log is only defined for strictly positive values
    non_positive = (df[COLUMNS_WITH_LOG] <= 0).sum()
    if non_positive.any():
        raise ValueError(f"Non-positive values, log impossible: {non_positive[non_positive > 0].to_dict()}")

    for col in COLUMNS_WITH_LOG:
        df[f"log_{col}"] = np.log(df[col])
    return df


def standardize(df, features):
    """Z-score of each feature (mean 0, std 1), with country identifiers kept for reading."""
    z = (df[features] - df[features].mean()) / df[features].std()
    return df[["ref_area", "name"]].join(z)


def extreme_values(df_z, features, threshold=Z_THRESHOLD):
    """One row per (country, feature) with |z| > threshold, most extreme first."""
    long = df_z.melt(id_vars=["ref_area", "name"], value_vars=features, var_name="feature", value_name="z")
    return long[long["z"].abs() > threshold].sort_values("z", key=abs, ascending=False)


def farthest_countries(df_z, features, n=5):
    """Countries farthest from the center of the cloud: Euclidean norm of their z-score vector (what k-means sees)."""
    distance = np.sqrt((df_z[features] ** 2).sum(axis=1))
    return df_z[["ref_area", "name"]].assign(distance=distance).nlargest(n, "distance")

## Standardisation

In [3]:
df_log = prepare_data()

ACTIVE_FEATURES = [
    "log_gdp_per_capita_ppp_current_ppp",
    "life_expectancy_yr",
    "log_co2_emissions_per_capita_t",
    "log_pm2_5_exposure_mcg_m3",
    "urban_population_percentage_pt",
]

df_standardized = standardize(df_log, ACTIVE_FEATURES)
df_standardized

,ref_area,name,log_gdp_per_capita_ppp_current_ppp,life_expectancy_yr,log_co2_emissions_per_capita_t,log_pm2_5_exposure_mcg_m3,urban_population_percentage_pt
0,AFG,Afghanistan,-1.675056,-0.981758,-1.373166,1.032759,-1.646080
1,AGO,Angola,-0.430299,-1.176619,-0.692904,0.024957,0.404746
2,ALB,Albania,0.350902,0.882613,-0.094694,-0.364574,-0.133206
3,ARE,United Arab Emirates,1.307965,1.337059,1.548266,1.673404,1.135664
4,ARG,Argentina,0.515495,0.579328,0.511685,-0.486794,1.437265
...,...,...,...,...,...,...,...
173,VNM,Viet Nam,-0.068454,0.193591,0.463766,0.367262,-1.061180
174,VUT,Vanuatu,-1.271359,-0.233921,-0.601341,-1.609171,-1.794322
176,ZAF,South Africa,-0.059279,-0.967467,0.849211,0.542389,0.115035
177,ZMB,Zambia,-1.159790,-0.938609,-0.906304,0.051063,-0.742376


In [4]:
df_standardized.describe().T

,count,mean,std,min,25%,50%,75%,max
log_gdp_per_capita_ppp_current_ppp,162.0,-4.276415e-16,1.0,-2.218231,-0.722306,0.065974,0.874760,1.858157
life_expectancy_yr,162.0,-5.811538e-16,1.0,-2.572115,-0.799540,0.114697,0.754166,1.508435
log_co2_emissions_per_capita_t,162.0,7.675616e-17,1.0,-2.633551,-0.677974,0.210873,0.734819,2.228335
log_pm2_5_exposure_mcg_m3,162.0,1.918904e-16,1.0,-2.275076,-0.591664,0.003754,0.732025,2.663157
urban_population_percentage_pt,162.0,-1.973730e-16,1.0,-2.127378,-0.833196,0.063297,0.817304,1.797974


In [5]:
# Univariate check: countries extreme on one feature
extreme_values(df_standardized, ACTIVE_FEATURES)

,ref_area,name,feature,z
612,QAT,Qatar,log_pm2_5_exposure_mcg_m3,2.663157
461,SOM,"Somalia, Fed. Rep.",log_co2_emissions_per_capita_t,-2.633551
273,NGA,Nigeria,life_expectancy_yr,-2.572115


In [6]:
# Multivariate check: countries far from the center on all features combined
farthest_countries(df_standardized, ACTIVE_FEATURES)

,ref_area,name,distance
137,QAT,Qatar,4.445833
119,NER,Niger,4.169526
27,CAF,Central African Republic,4.048484
10,BDI,Burundi,4.006077
156,TCD,Chad,3.976778


### Extremes check (main version, standardized, 162 countries)

**Observation.** Only 3 countries exceed |z| > 2.5, each on a single feature: Nigeria (life expectancy, −2.57), Somalia (log CO₂, −2.63), Qatar (log PM2.5, +2.66). On all features combined, Qatar is the farthest from the center (distance 4.4, high on every feature), followed by Niger, Central African Republic, Burundi and Chad (≈ 4.0).

**Conclusion.** The log transform worked: no extreme value on several features at once, all countries kept. Qatar is the most isolated country of the cloud: watch whether it forms a cluster on its own.

## Variants (clustering stage)

| Variant | Change from the main version |
|---|---|
| A | + health % of GDP |
| B | Pollution only: log CO₂ + log PM2.5 |
| C | CO₂ and PM2.5 without log |
| D | 177 countries: micro-states put back, except Nauru and Tuvalu (zero CO₂, log impossible) |
| E1 / E2 | − CO₂ / − urbanization |

_Note: the z-score of a feature only depends on its own column and on the set of countries. Variants B, E1 and E2 therefore show the same univariate extremes as the main version; only the multivariate distances change._

In [7]:
VARIANTS = {
    "A - with health %": ACTIVE_FEATURES + ["health_expenditure_percentage_of_gdp_pt_gdp"],
    "B - pollution only": ["log_co2_emissions_per_capita_t", "log_pm2_5_exposure_mcg_m3"],
    "C - raw CO2 and PM2.5": [
        "log_gdp_per_capita_ppp_current_ppp",
        "life_expectancy_yr",
        "co2_emissions_per_capita_t",
        "pm2_5_exposure_mcg_m3",
        "urban_population_percentage_pt",
    ],
    "E1 - without CO2": [f for f in ACTIVE_FEATURES if f != "log_co2_emissions_per_capita_t"],
    "E2 - without urbanization": [f for f in ACTIVE_FEATURES if f != "urban_population_percentage_pt"],
}

for variant, features in VARIANTS.items():
    df_z = standardize(df_log, features)
    print(f"=== Variant {variant} ({len(features)} features)")
    display(extreme_values(df_z, features))
    display(farthest_countries(df_z, features))

=== Variant A - with health % (6 features)


,ref_area,name,feature,z
965,USA,United States,health_expenditure_percentage_of_gdp_pt_gdp,3.601256
810,AFG,Afghanistan,health_expenditure_percentage_of_gdp_pt_gdp,2.994966
612,QAT,Qatar,log_pm2_5_exposure_mcg_m3,2.663157
461,SOM,"Somalia, Fed. Rep.",log_co2_emissions_per_capita_t,-2.633551
273,NGA,Nigeria,life_expectancy_yr,-2.572115


,ref_area,name,distance
137,QAT,Qatar,4.669464
170,USA,United States,4.527150
27,CAF,Central African Republic,4.303090
0,AFG,Afghanistan,4.289666
119,NER,Niger,4.262706


=== Variant B - pollution only (2 features)


,ref_area,name,feature,z
288,QAT,Qatar,log_pm2_5_exposure_mcg_m3,2.663157
137,SOM,"Somalia, Fed. Rep.",log_co2_emissions_per_capita_t,-2.633551


,ref_area,name,distance
137,QAT,Qatar,3.472446
119,NER,Niger,2.716961
34,COD,"Congo, Dem. Rep.",2.648545
148,SOM,"Somalia, Fed. Rep.",2.634962
141,SAU,Saudi Arabia,2.592329


... (5 outputs left out)

,ref_area,name,distance
137,QAT,Qatar,3.847071
119,NER,Niger,3.678814
156,TCD,Chad,3.541424
27,CAF,Central African Republic,3.259521
10,BDI,Burundi,3.243830


=== Variant E2 - without urbanization (4 features)


,ref_area,name,feature,z
612,QAT,Qatar,log_pm2_5_exposure_mcg_m3,2.663157
461,SOM,"Somalia, Fed. Rep.",log_co2_emissions_per_capita_t,-2.633551
273,NGA,Nigeria,life_expectancy_yr,-2.572115


,ref_area,name,distance
137,QAT,Qatar,4.080525
27,CAF,Central African Republic,3.967878
148,SOM,"Somalia, Fed. Rep.",3.834361
119,NER,Niger,3.656931
156,TCD,Chad,3.637866


In [8]:
# Variant D: micro-states put back, except Nauru and Tuvalu (zero CO₂ since 2000, log impossible)
ZERO_CO2_AREAS = ["NRU", "TUV"]
df_log_d = prepare_data(min_population=0, excluded_areas=ZERO_CO2_AREAS)

df_z_d = standardize(df_log_d, ACTIVE_FEATURES)
print(f"=== Variant D - micro-states back ({len(df_log_d)} countries)")
display(extreme_values(df_z_d, ACTIVE_FEATURES))
display(farthest_countries(df_z_d, ACTIVE_FEATURES))

=== Variant D - micro-states back (177 countries)


,ref_area,name,feature,z
410,FSM,"Micronesia, Fed. Sts.",log_co2_emissions_per_capita_t,-4.393428
461,MHL,Marshall Islands,log_co2_emissions_per_capita_t,-4.158723
297,NGA,Nigeria,life_expectancy_yr,-2.641706
667,QAT,Qatar,log_pm2_5_exposure_mcg_m3,2.611223
332,TCD,Chad,life_expectancy_yr,-2.555466


,ref_area,name,distance
56,FSM,"Micronesia, Fed. Sts.",5.467686
107,MHL,Marshall Islands,4.883910
137,QAT,Qatar,4.382827
119,NER,Niger,4.053317
27,CAF,Central African Republic,3.948647


### Extremes check (variants)

**Observation.**
- **A (+ health %):** the USA has the highest value on the added feature (z = 3.60), ahead of Afghanistan (z = 2.99). Both enter the five farthest countries (2nd and 4th), although their profiles are opposite on the other five features.
- **B (pollution only):** Qatar remains the farthest country (distance 3.47, ahead of Niger at 2.72).
- **C (raw CO₂ and PM2.5):** Qatar becomes far more extreme (z = 7.44 on CO₂, 5.22 on PM2.5; distance 9.50, about twice Bahrain's 4.62). Gulf countries fill the top four.
- **D (micro-states back, 177 countries):** Micronesia and the Marshall Islands become the farthest countries (distances 5.47 and 4.88), ahead of Qatar (4.38), because of extreme log CO₂ values (z = −4.39 and −4.16).
- **E1 (without CO₂):** only two extreme values remain (Qatar, Nigeria), the fewest with B.
- **E2 (without urbanization):** close to E1, with four of the five farthest countries in common. Somalia stays extreme, on CO₂.

**Conclusion.**
- **A:** adding one feature reshuffles the farthest countries, and a high value can describe opposite profiles. This supports keeping it out of the main version.
- **B:** a simple pollution-focused view, worth developing as its own analysis.
- **C:** without the log, the tails dominate and the Gulf countries stand out. This supports the log transform.
- **D:** as expected, micro-states return among the extreme values, notably on CO₂. The effect on the clusters remains to be checked.
- **E1:** promising, since CO₂ correlates strongly with GDP per capita (Spearman 0.89). To be compared with the main version at the clustering stage.

_Distances are only comparable between variants with the same number of features._